# 01 — Ingest Raw Data into Bronze Delta Table

**Ticket:** I-01 
**Description:** Ingest raw NYC Yellow Taxi trip data into a Bronze Delta table (append-only, no transforms). 
**Source:** `data_academy_resources.nyc_taxi.yellow_tripdata` 
**Target:** `students_data.chris-foreman.bronze_yellow_tripdata`

### Approach
- Read the full source table as-is (no schema changes, no filtering, no cleaning)
- Add a single ingestion metadata column (`_ingested_at`) for lineage tracking
- Write to Delta in **append** mode so the notebook is re-runnable for incremental loads

## Setup

In [0]:
from src.constants import BRONZE_TABLE, SOURCE_TABLE
from src.ingest import add_ingestion_metadata, write_bronze

## Read raw data

In [0]:
# Read source table — no transforms, preserving all original columns and types
df_raw = spark.read.table(SOURCE_TABLE)

# Add ingestion metadata for lineage tracking
df_bronze = add_ingestion_metadata(df_raw)

## Write Bronze Delta table

In [0]:
# Write to Bronze Delta table (append-only)
write_bronze(df_bronze, BRONZE_TABLE)

print(f"✅ Bronze ingestion complete: {BRONZE_TABLE}")

## Verify

In [0]:
# --- Verification ---
df_verify = spark.read.table(BRONZE_TABLE)

source_count = spark.read.table(SOURCE_TABLE).count()
bronze_count = df_verify.count()

print(f"Source row count:  {source_count:,}")
print(f"Bronze row count:  {bronze_count:,}")
print(f"Match: {'✅' if bronze_count >= source_count else '❌'}")
print("\nBronze schema:")
df_verify.printSchema()
print("\nSample rows:")
display(df_verify.limit(5))